# LeetCode 3695 — Maximize Alternating Sum Using Swaps bằng Python

- Link: https://leetcode.com/problems/maximize-alternating-sum-using-swaps/
- Độ khó: **Hard**
- Chủ đề: **Union-Find (DSU)**, thành phần liên thông, hoán đổi tuỳ ý trong thành phần ⇒ **mọi hoán vị**, tham lam sắp xếp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxAlternatingSum(self, nums: List[int], swaps: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Hiểu vì sao các hoán đổi (dùng bao nhiêu lần tuỳ ý) trên một **thành phần liên thông** tạo ra **mọi hoán vị** của các phần tử trong thành phần đó.
2. Gom chỉ số thành thành phần bằng DSU (hoặc BFS).
3. Trong mỗi thành phần có `e` vị trí **chẵn**: đặt `e` giá trị lớn nhất vào đó (cộng), phần còn lại vào vị trí lẻ (trừ).
4. Chứng minh tham lam bằng lập luận hoán đổi.
5. Kiểm chứng bằng BFS trên mọi trạng thái đạt được.

## 2. Đề bài & ví dụ

Tổng xen kẽ: `nums[0] − nums[1] + nums[2] − …`. Được hoán đổi `nums[p]` và `nums[q]` với mọi cặp `[p, q]` trong `swaps`, bao nhiêu lần, theo thứ tự nào cũng được. Tìm tổng xen kẽ lớn nhất.

**Ví dụ 1:** `nums = [1,2,3], swaps = [[0,2],[1,2]]` → `4`.

**Ví dụ 2:** `nums = [1,2,3], swaps = [[1,2]]` → `2`.

**Ví dụ 3:** `nums = [1,10^9,1,10^9,1,10^9], swaps = []` → `−2999999997`.

**Ràng buộc:** `n, |swaps| ≤ 10^5`, `nums[i] ≤ 10^9`.

In [1]:
from typing import List
from collections import deque, defaultdict
from heapq import nlargest
import random, time

examples = [
    ([1, 2, 3], [[0, 2], [1, 2]], 4),
    ([1, 2, 3], [[1, 2]], 2),
    ([1, 10**9, 1, 10**9, 1, 10**9], [], -2999999997),
]
for a, sw, exp in examples:
    print(a, sw, "-> kỳ vọng", exp)

[1, 2, 3] [[0, 2], [1, 2]] -> kỳ vọng 4
[1, 2, 3] [[1, 2]] -> kỳ vọng 2
[1, 1000000000, 1, 1000000000, 1, 1000000000] [] -> kỳ vọng -2999999997


## 3. Kiến thức nền

### 3.1. Hoán vị chuyển vị sinh ra mọi hoán vị trên thành phần liên thông

Trên một cây khung của thành phần, có thể "đưa" bất kỳ phần tử nào tới bất kỳ vị trí nào mà các vị trí khác trở lại như cũ (chuyển vị `(a b)` được tạo từ các chuyển vị dọc đường đi a → b). Mọi hoán vị là tích các chuyển vị ⇒ đạt được mọi sắp xếp trong thành phần.

In [2]:
def reachable(nums, swaps):
    start = tuple(nums); seen = {start}; q = deque([start])
    while q:
        cur = q.popleft()
        for p, r in swaps:
            nxt = list(cur); nxt[p], nxt[r] = nxt[r], nxt[p]; nxt = tuple(nxt)
            if nxt not in seen: seen.add(nxt); q.append(nxt)
    return seen

print(len(reachable([1, 2, 3, 4], [[0, 1], [1, 2], [2, 3]])), "= 4! = 24 (đường thẳng sinh mọi hoán vị)")
print(sorted(reachable([1, 2, 3], [[0, 2]])))

24 = 4! = 24 (đường thẳng sinh mọi hoán vị)
[(1, 2, 3), (3, 2, 1)]


### 3.2. Union-Find

`find` với nén đường, `union` theo kích thước: gần như `O(1)` mỗi thao tác. Dùng để gom chỉ số theo thành phần.

In [3]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n)); self.sz = [1] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b: return
        if self.sz[a] < self.sz[b]: a, b = b, a
        self.p[b] = a; self.sz[a] += self.sz[b]

d = DSU(5); d.union(0, 2); d.union(2, 4)
print([d.find(i) for i in range(5)])

[0, 1, 0, 3, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mỗi thành phần là một "túi" giá trị cùng một tập vị trí cố định. Vị trí chẵn mang dấu `+`, lẻ mang dấu `−`. Muốn tổng lớn nhất ⇒ giá trị **lớn** vào vị trí `+`, giá trị **nhỏ** vào vị trí `−`.

Ví dụ 1: cả ba chỉ số một thành phần; vị trí chẵn `{0, 2}`, lẻ `{1}`; giá trị `{1,2,3}` ⇒ `3 + 2 − 1 = 4`. ✔

## 5. Cách 1 — Brute Force: BFS mọi trạng thái đạt được

In [4]:
def alt(a):
    return sum(x if i % 2 == 0 else -x for i, x in enumerate(a))

def brute(nums, swaps):
    return max(alt(s) for s in reachable(nums, swaps))

for a, sw, exp in examples:
    print(brute(a, sw), "kỳ vọng", exp)

4 kỳ vọng 4
2 kỳ vọng 2
-2999999997 kỳ vọng -2999999997


**Nút thắt:** số trạng thái tới `n!`.

## 6. Ý tưởng thuật toán / bước đột phá

1. DSU gom các chỉ số theo `swaps`.
2. Với mỗi thành phần: `vals` = giá trị, `e` = số chỉ số chẵn.
3. Sắp `vals` giảm dần: đóng góp `= Σ vals[:e] − Σ vals[e:]`.
4. Cộng mọi thành phần.

`O((n + |swaps|) α(n) + n log n)`.

## 7. Các cách tiếp theo

- **Cách 2:** BFS/DFS tìm thành phần + sắp xếp.
- **Cách 3:** DSU + sắp xếp (lời giải nộp).
- **Cách 4:** DSU + `heapq.nlargest(e, vals)` — chỉ cần `e` phần tử lớn nhất, không cần sắp toàn bộ.

### 7.1. Cách 2 — BFS tìm thành phần

In [5]:
def bfs_components(nums, swaps):
    n = len(nums); adj = [[] for _ in range(n)]
    for p, q in swaps: adj[p].append(q); adj[q].append(p)
    seen = [False] * n; total = 0
    for s in range(n):
        if seen[s]: continue
        comp = [s]; seen[s] = True
        for u in comp:
            for v in adj[u]:
                if not seen[v]: seen[v] = True; comp.append(v)
        vals = sorted((nums[i] for i in comp), reverse=True)
        e = sum(1 for i in comp if i % 2 == 0)
        total += sum(vals[:e]) - sum(vals[e:])
    return total

for a, sw, exp in examples:
    print(bfs_components(a, sw), "kỳ vọng", exp)

4 kỳ vọng 4
2 kỳ vọng 2
-2999999997 kỳ vọng -2999999997


### 7.2. Cách 3 — DSU + sắp xếp (lời giải chính)

In [6]:
def dsu_sort(nums, swaps):
    n = len(nums); d = DSU(n)
    for p, q in swaps: d.union(p, q)
    groups = defaultdict(list); evens = defaultdict(int)
    for i in range(n):
        r = d.find(i); groups[r].append(nums[i]); evens[r] += (i % 2 == 0)
    total = 0
    for r, vals in groups.items():
        vals.sort(reverse=True); e = evens[r]
        total += sum(vals[:e]) - sum(vals[e:])
    return total

for a, sw, exp in examples:
    print(dsu_sort(a, sw), "kỳ vọng", exp)

4 kỳ vọng 4
2 kỳ vọng 2
-2999999997 kỳ vọng -2999999997


### 7.3. Cách 4 — DSU + `nlargest`

Đóng góp = `2·Σ(top e) − Σ(vals)`. `nlargest(e, vals)` tốn `O(|vals| log e)`.

In [7]:
def dsu_nlargest(nums, swaps):
    n = len(nums); d = DSU(n)
    for p, q in swaps: d.union(p, q)
    groups = defaultdict(list); evens = defaultdict(int)
    for i in range(n):
        r = d.find(i); groups[r].append(nums[i]); evens[r] += (i % 2 == 0)
    return sum(2 * sum(nlargest(evens[r], vals)) - sum(vals) for r, vals in groups.items())

for a, sw, exp in examples:
    print(dsu_nlargest(a, sw), "kỳ vọng", exp)

4 kỳ vọng 4
2 kỳ vọng 2
-2999999997 kỳ vọng -2999999997


## 8. TRACE từng bước

In [8]:
def trace(nums, swaps):
    n = len(nums); d = DSU(n)
    for p, q in swaps: d.union(p, q)
    groups = defaultdict(list)
    for i in range(n): groups[d.find(i)].append(i)
    total = 0
    for r, idx in groups.items():
        vals = sorted((nums[i] for i in idx), reverse=True); e = sum(1 for i in idx if i % 2 == 0)
        c = sum(vals[:e]) - sum(vals[e:]); total += c
        print(f"thành phần {idx}: giá trị giảm dần {vals}, {e} vị trí chẵn -> +{vals[:e]} -{vals[e:]} = {c}")
    print("tổng:", total, "| brute:", brute(nums, swaps))

trace([5, 1, 4, 2, 3, 6], [[0, 3], [3, 5], [1, 2]])

thành phần [0, 3, 5]: giá trị giảm dần [6, 5, 2], 1 vị trí chẵn -> +[6] -[5, 2] = -1
thành phần [1, 2]: giá trị giảm dần [4, 1], 1 vị trí chẵn -> +[4] -[1] = 3
thành phần [4]: giá trị giảm dần [3], 1 vị trí chẵn -> +[3] -[] = 3
tổng: 5 | brute: 5


In [9]:
# Một cách sắp xếp cụ thể đạt tối ưu
def construct(nums, swaps):
    n = len(nums); d = DSU(n)
    for p, q in swaps: d.union(p, q)
    groups = defaultdict(list)
    for i in range(n): groups[d.find(i)].append(i)
    res = [0] * n
    for idx in groups.values():
        vals = sorted((nums[i] for i in idx), reverse=True)
        ev = [i for i in idx if i % 2 == 0]; od = [i for i in idx if i % 2]
        for i, v in zip(ev + od, vals): res[i] = v
    return res

arr = construct([5, 1, 4, 2, 3, 6], [[0, 3], [3, 5], [1, 2]])
print("mảng tối ưu:", arr, "| tổng xen kẽ:", alt(arr), "| đạt được?", tuple(arr) in reachable([5, 1, 4, 2, 3, 6], [[0, 3], [3, 5], [1, 2]]))

mảng tối ưu: [6, 1, 4, 5, 3, 2] | tổng xen kẽ: 5 | đạt được? True


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def maxAlternatingSum(self, nums: List[int], swaps: List[List[int]]) -> int:
        n = len(nums)
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for p, q in swaps:
            a, b = find(p), find(q)
            if a != b:
                parent[a] = b
        groups = {}
        evens = {}
        for i in range(n):
            r = find(i)
            groups.setdefault(r, []).append(nums[i])
            evens[r] = evens.get(r, 0) + (1 - (i & 1))
        total = 0
        for r, vals in groups.items():
            vals.sort(reverse=True)
            e = evens[r]
            total += sum(vals[:e]) - sum(vals[e:])
        return total

sol = Solution()
for a, sw, exp in examples:
    got = sol.maxAlternatingSum(a, sw)
    print(got, "✔" if got == exp else "✘")

4 ✔
2 ✔
-2999999997 ✔


## 10. Vì sao thuật toán đúng

1. **Không gian đạt được** = mọi mảng thu được bằng cách hoán vị tuỳ ý **trong từng thành phần** (3.1); các phần tử không rời khỏi thành phần của mình (mỗi hoán đổi giữ nguyên tập giá trị của thành phần).
2. **Tách rời:** tổng xen kẽ = tổng theo thành phần của `Σ_{i chẵn} − Σ_{i lẻ}` ⇒ tối ưu từng thành phần độc lập.
3. **Tham lam trong thành phần:** giả sử một sắp xếp tối ưu có `x` ở vị trí lẻ, `y` ở vị trí chẵn với `x > y`. Đổi chỗ chúng làm tổng thay đổi `(x − y) − (y − x) = 2(x − y) > 0` — mâu thuẫn. Vậy mọi giá trị ở vị trí chẵn `≥` mọi giá trị ở vị trí lẻ ⇒ vị trí chẵn nhận đúng `e` giá trị lớn nhất (giá trị bằng nhau không ảnh hưởng tổng).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS trạng thái | `O(n! · |swaps| · n)` | `O(n!)` |
| 2. BFS thành phần + sắp | `O(n + |swaps| + n log n)` | `O(n + |swaps|)` |
| 3. DSU + sắp | `O(|swaps| α(n) + n log n)` | `O(n)` |
| 4. DSU + `nlargest` | `O(|swaps| α(n) + n log n)` | `O(n)` |

## 12. Bộ test

In [11]:
METHODS = [brute, bfs_components, dsu_sort, dsu_nlargest, lambda a, s: Solution().maxAlternatingSum(a, s)]
for a, sw, exp in examples:
    assert all(f(a, sw) == exp for f in METHODS[1:]) and (len(a) > 8 or brute(a, sw) == exp)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên
assert Solution().maxAlternatingSum([5, 7], []) == -2
assert Solution().maxAlternatingSum([5, 7], [[0, 1]]) == 2
assert Solution().maxAlternatingSum([3, 3, 3], [[0, 1], [1, 2]]) == 3
print("Ca biên: OK")

Ca biên: OK


In [13]:
# 12.3 ngẫu nhiên so với BFS trạng thái
rng = random.Random(3695)
for _ in range(600):
    n = rng.randint(2, 7); a = [rng.randint(1, 9) for _ in range(n)]
    pairs = [[p, q] for p in range(n) for q in range(p + 1, n)]
    sw = rng.sample(pairs, rng.randint(0, min(len(pairs), 5)))
    b = brute(a, sw)
    assert all(f(a, sw) == b for f in METHODS[1:]), (a, sw)
print("600 test ngẫu nhiên: OK")

600 test ngẫu nhiên: OK


In [14]:
# 12.4 test lớn
rng = random.Random(1)
n = 10**5
a = [rng.randint(1, 10**9) for _ in range(n)]
sw = []
seen = set()
while len(sw) < 10**5:
    p = rng.randrange(n - 1); q = rng.randrange(p + 1, n)
    if (p, q) not in seen: seen.add((p, q)); sw.append([p, q])
t0 = time.time(); x = Solution().maxAlternatingSum(a, sw); t1 = time.time()
print("đáp án:", x, "| thời gian:", round(t1 - t0, 2), "s | khớp nlargest:", x == dsu_nlargest(a, sw))

đáp án: 19254201341679 | thời gian: 0.13 s | khớp nlargest: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ thực hiện mỗi hoán đổi **một lần** (theo thứ tự cho trước) khi có lợi

In [15]:
def wrong_one_pass(nums, swaps):
    a = nums[:]
    for p, q in swaps:
        b = a[:]; b[p], b[q] = b[q], b[p]
        if alt(b) > alt(a): a = b                                  # SAI: tham lam cục bộ
    return alt(a)

print("sai:", wrong_one_pass([1, 2, 3], [[0, 2], [1, 2]]), "| đúng:", brute([1, 2, 3], [[0, 2], [1, 2]]))

sai: 2 | đúng: 4


### Lỗi 2 — Coi mọi chỉ số chẵn/lẻ "toàn cục" (bỏ qua thành phần)

In [16]:
def wrong_global(nums, swaps):
    vals = sorted(nums, reverse=True); e = (len(nums) + 1) // 2
    return sum(vals[:e]) - sum(vals[e:])                            # SAI: như thể đổi được mọi cặp

print("sai:", wrong_global([1, 2, 3], [[1, 2]]), "| đúng:", brute([1, 2, 3], [[1, 2]]))

sai: 4 | đúng: 2


### Lỗi 3 — Chỉ xét các chỉ số **xuất hiện trong `swaps`**

Chỉ số không có trong `swaps` là thành phần một phần tử — vẫn phải cộng/trừ giá trị của nó.

In [17]:
def wrong_skip_isolated(nums, swaps):
    used = {i for p in swaps for i in p}
    return dsu_sort([nums[i] if i in used else 0 for i in range(len(nums))], swaps)   # SAI

print("sai:", wrong_skip_isolated([1, 2, 3], [[1, 2]]), "| đúng:", brute([1, 2, 3], [[1, 2]]))

sai: 1 | đúng: 2


### Lỗi 4 — Đếm số vị trí chẵn của thành phần bằng `ceil(size/2)`

Số vị trí chẵn phụ thuộc **chỉ số thật**, không phải kích thước thành phần.

In [18]:
def wrong_even_count(nums, swaps):
    n = len(nums); d = DSU(n)
    for p, q in swaps: d.union(p, q)
    g = defaultdict(list)
    for i in range(n): g[d.find(i)].append(nums[i])
    t = 0
    for vals in g.values():
        vals.sort(reverse=True); e = (len(vals) + 1) // 2           # SAI
        t += sum(vals[:e]) - sum(vals[e:])
    return t

print("sai:", wrong_even_count([1, 5, 2, 7], [[1, 3]]), "| đúng:", brute([1, 5, 2, 7], [[1, 3]]))

sai: 5 | đúng: -9


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hoán đổi các cặp cho trước, bao nhiêu lần cũng được" ⇒ **thành phần liên thông** ⇒ sắp xếp tự do trong thành phần (LC 1202 *Smallest String With Swaps*, LC 1722).
- Hàm mục tiêu tách theo vị trí với hệ số `±1` ⇒ ghép lớn–với–`+`, nhỏ–với–`−` (bất đẳng thức hoán vị).
- DSU là công cụ nhanh nhất để gom chỉ số.

## 15. Biến thể của bài

1. **Hệ số tuỳ ý** `c_i` thay vì `±1`: trong mỗi thành phần, ghép giá trị lớn với hệ số lớn (bất đẳng thức sắp xếp lại).
2. **LC 1202**: chuỗi nhỏ nhất theo từ điển — sắp ký tự tăng dần theo chỉ số tăng trong thành phần.
3. **Tối thiểu số lần hoán đổi** để đạt cấu hình tối ưu: đếm chu trình của hoán vị cần thực hiện.
4. **Hoán đổi chỉ kề nhau** (bubble): mọi hoán vị của cả mảng ⇒ trường hợp đặc biệt.

In [19]:
# Biến thể 1: hệ số tuỳ ý — ghép theo thứ tự (bất đẳng thức sắp xếp lại)
def max_weighted(nums, coef, swaps):
    n = len(nums); d = DSU(n)
    for p, q in swaps: d.union(p, q)
    g = defaultdict(list)
    for i in range(n): g[d.find(i)].append(i)
    total = 0
    for idx in g.values():
        vals = sorted(nums[i] for i in idx); cs = sorted(coef[i] for i in idx)
        total += sum(v * c for v, c in zip(vals, cs))
    return total

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 6); a = [rng.randint(-5, 9) for _ in range(n)]; c = [rng.randint(-3, 3) for _ in range(n)]
    pairs = [[p, q] for p in range(n) for q in range(p + 1, n)]; sw = rng.sample(pairs, rng.randint(0, min(len(pairs), 4)))
    best = max(sum(x * y for x, y in zip(s, c)) for s in reachable(a, sw))
    assert max_weighted(a, c, sw) == best
print("Biến thể 1 OK")

Biến thể 1 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. BFS trạng thái | Liệt kê | `O(n!)` | Kiểm chứng |
| 2. BFS thành phần | Đồ thị kề | `O(n log n + E)` | Không quen DSU |
| 3. DSU + sắp | Gom + tham lam | `O(n log n + E α)` | **Lời giải chuẩn** |
| 4. DSU + `nlargest` | Chỉ lấy top `e` | `O(n log n + E α)` | Khi `e` nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [4,9,1], swaps = [[0,1]]`.

**Bài 2 (tính tay).** `nums = [2,8,6,1,5], swaps = [[0,4],[1,3]]`.

**Bài 3 (điền TODO).** Viết `min_alternating_sum(nums, swaps)` — tổng xen kẽ **nhỏ nhất**.

**Bài 4 (tự viết lại).** Viết `smallest_string_with_swaps(s, pairs)` (LC 1202).

**Bài 5 (mini challenge).** Giả sử các giá trị phân biệt và được phép hoán đổi **bất kỳ cặp nào trong cùng thành phần**. Tính số hoán đổi tối thiểu để biến `nums` thành **đúng** cấu hình tối ưu do hàm `construct` (mục 8) tạo ra.

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_alternating_sum(nums, swaps):
    # TODO: đặt giá trị NHỎ vào vị trí chẵn
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def min_swaps_to_optimal(nums, swaps):
    # TODO: trong mỗi thành phần, số hoán đổi = kích thước - số chu trình
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Thành phần `{0,1}` và `{2}`. (2) `{0,1}`: 9 vào chỉ số 0, 4 vào chỉ số 1 ⇒ `5`. (3) Chỉ số 2 chẵn: `+1` ⇒ `6`.

**Bài 2.** (1) `{0,4}` (đều chẵn): `+2 +5`. (2) `{1,3}` (đều lẻ): `−8 −1`. (3) `{2}`: `+6` ⇒ `4`.

**Bài 3.** (1) Đối xứng. (2) Sắp tăng dần, `e` nhỏ nhất vào chỉ số chẵn. (3) Hoặc `−maxAlternatingSum(−nums)`.

**Bài 4.** (1) DSU trên chỉ số. (2) Mỗi thành phần: sắp ký tự, sắp chỉ số. (3) Ghép theo thứ tự.

**Bài 5.** (1) Cấu hình đích từ `construct`. (2) Hoán vị đích trên từng thành phần. (3) `Σ (kích thước chu trình − 1)`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().maxAlternatingSum([4, 9, 1], [[0, 1]]) == 6 == brute([4, 9, 1], [[0, 1]])
assert Solution().maxAlternatingSum([2, 8, 6, 1, 5], [[0, 4], [1, 3]]) == 4 == brute([2, 8, 6, 1, 5], [[0, 4], [1, 3]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def min_alternating_sum_sol(nums, swaps):
    return -Solution().maxAlternatingSum([-x for x in nums], swaps)

def smallest_string_with_swaps_sol(s, pairs):
    n = len(s); d = DSU(n)
    for a, b in pairs: d.union(a, b)
    g = defaultdict(list)
    for i in range(n): g[d.find(i)].append(i)
    res = list(s)
    for idx in g.values():
        for i, ch in zip(sorted(idx), sorted(s[i] for i in idx)): res[i] = ch
    return "".join(res)

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 6); a = [rng.randint(1, 9) for _ in range(n)]
    pairs = [[p, q] for p in range(n) for q in range(p + 1, n)]; sw = rng.sample(pairs, rng.randint(0, min(len(pairs), 4)))
    assert min_alternating_sum_sol(a, sw) == min(alt(s) for s in reachable(a, sw))
assert smallest_string_with_swaps_sol("dcab", [[0, 3], [1, 2]]) == "bacd"
assert smallest_string_with_swaps_sol("dcab", [[0, 3], [1, 2], [0, 2]]) == "abcd"
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5: số hoán đổi tối thiểu (hoán đổi tự do trong thành phần) để đạt cấu hình tối ưu, giá trị phân biệt
def min_swaps_to_optimal_sol(nums, swaps):
    target = construct(nums, swaps)
    pos = {v: i for i, v in enumerate(target)}
    perm = [pos[v] for v in nums]                       # phần tử ở i phải tới perm[i]
    seen = [False] * len(nums); res = 0
    for i in range(len(nums)):
        if not seen[i]:
            L = 0; j = i
            while not seen[j]: seen[j] = True; j = perm[j]; L += 1
            res += L - 1
    return res

def min_swaps_brute(nums, swaps):
    n = len(nums); d = DSU(n)
    for p, q in swaps: d.union(p, q)
    free = [[p, q] for p in range(n) for q in range(p + 1, n) if d.find(p) == d.find(q)]
    target = tuple(construct(nums, swaps)); start = tuple(nums); dist = {start: 0}; dq = deque([start])
    while dq:
        cur = dq.popleft()
        if cur == target: return dist[cur]
        for p, q in free:
            nx = list(cur); nx[p], nx[q] = nx[q], nx[p]; nx = tuple(nx)
            if nx not in dist: dist[nx] = dist[cur] + 1; dq.append(nx)

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(2, 6); a = rng.sample(range(1, 20), n)
    pairs = [[p, q] for p in range(n) for q in range(p + 1, n)]; sw = rng.sample(pairs, rng.randint(0, min(len(pairs), 4)))
    assert min_swaps_to_optimal_sol(a, sw) == min_swaps_brute(a, sw)
print("Bài 5 OK: số hoán đổi = Σ (độ dài chu trình − 1), khớp BFS")

Bài 5 OK: số hoán đổi = Σ (độ dài chu trình − 1), khớp BFS


**Mở rộng Bài 5:** nếu hỏi số hoán đổi tối thiểu tới **bất kỳ** cấu hình tối ưu nào (khi nhiều cấu hình cùng tối ưu, ví dụ thứ tự các giá trị giữa các vị trí chẵn không quan trọng), cần chọn đích sao cho số chu trình **lớn nhất** — bài toán khó hơn, để bạn tự đào sâu.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
DSU: union mọi cặp trong swaps
for mỗi thành phần:
    vals = sorted(giá trị, reverse=True); e = số chỉ số CHẴN trong thành phần
    total += sum(vals[:e]) - sum(vals[e:])
```

**Checklist:**

- [ ] Gom theo thành phần liên thông (không phải toàn cục)?
- [ ] `e` đếm theo **chỉ số thật** chẵn?
- [ ] Không bỏ sót chỉ số không xuất hiện trong `swaps`?
- [ ] Kết quả có thể âm (ví dụ 3)?
- [ ] Đã so với BFS trạng thái?